[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oalnaseri/mobilcom_course/blob/main/03_reflection_two_path_interactive.ipynb)

> **Run this notebook in Google Colab** — click the badge above (no local install needed).
> The setup cell below installs dependencies and enables interactive `ipywidgets` sliders in Colab.


# 📡 Reflection & Two-Path Model
### Interactive Teaching Notebook — DHBW Mobile Communications
---
> **Key idea:** When a direct wave and a ground-reflected wave reach the same receiver,  
> they superpose. Depending on their relative **phase difference**, the received power  
> can be **stronger** (constructive) or **much weaker** (destructive) than with the direct path alone.

**This notebook has three interactive parts:**
1. **Scene viewer** — geometry of the two-path model, phase difference, and phasors
2. **Received power vs. distance** — see the oscillations, breakpoint, and 40 dB/decade roll-off
3. **Wave superposition animation** — watch the two waves add or cancel in real time


In [ ]:
# ============================================================
#  COLAB / ENVIRONMENT SETUP  (safe to run locally too)
# ============================================================
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # numpy + matplotlib ship with Colab; ensure ipywidgets is present
    !pip install -q ipywidgets

    # Required so ipywidgets sliders / interactive output render in Colab
    from google.colab import output
    output.enable_custom_widget_manager()
    print('✅ Colab detected — custom widget manager enabled.')
else:
    print('✅ Running locally (Jupyter) — no extra setup needed.')

# Inline backend works reliably for slider-driven redraws in both envs
%matplotlib inline


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Arc, FancyBboxPatch
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')
print("✅ Libraries loaded.")


In [ ]:
# PHYSICS HELPERS
c0 = 3e8   # speed of light [m/s]

def two_path_power(f_Hz, d_m, h_tx_m, h_rx_m, Pt_dBm=0, Gt_dBi=0, Gr_dBi=0,
                    refl_coeff=-1.0):
    """
    Two-path (ground reflection) received power in dBm.
    Returns (Prx_two_path_dBm, Prx_los_dBm, phase_diff_rad, d1, d2)
    """
    lam = c0 / f_Hz

    # Direct path length
    d1 = np.sqrt(d_m**2 + (h_tx_m - h_rx_m)**2)

    # Reflected path length (image method)
    d2 = np.sqrt(d_m**2 + (h_tx_m + h_rx_m)**2)

    # Free-space path loss for direct path
    FSPL1 = (4 * np.pi * d1 / lam)**2
    E1 = np.sqrt(1.0 / FSPL1)            # relative field amplitude

    # Reflected path
    FSPL2 = (4 * np.pi * d2 / lam)**2
    E2 = np.sqrt(1.0 / FSPL2) * abs(refl_coeff)

    # Phase of each wave at receiver
    phi1 = 2 * np.pi * d1 / lam
    phi2 = 2 * np.pi * d2 / lam + np.pi  # +pi for perfect ground (Gamma = -1)

    # Superposition (complex phasors)
    E_total = E1 * np.exp(1j * phi1) + E2 * np.exp(1j * phi2)
    P_two = abs(E_total)**2

    # LOS only
    P_los = E1**2

    # Convert to dBm (relative to 1 mW isotropic reference)
    ref = (lam / (4 * np.pi))**2
    Prx_two = Pt_dBm + Gt_dBi + Gr_dBi + 10*np.log10(P_two / (1/(4*np.pi*1)**2) * ref + 1e-300)
    Prx_los  = Pt_dBm + Gt_dBi + Gr_dBi + 10*np.log10(P_los  / (1/(4*np.pi*1)**2) * ref + 1e-300)

    phase_diff = (phi2 - phi1) % (2*np.pi)
    return Prx_two, Prx_los, phase_diff, d1, d2, E1, E2, phi1, phi2

def breakpoint_distance(f_Hz, h_tx, h_rx):
    """Breakpoint distance: d_BP = 4*h_tx*h_rx/lambda"""
    lam = c0 / f_Hz
    return 4 * h_tx * h_rx / lam

print("✅ Physics helpers defined.")


In [ ]:
# VISUALISATION 1 — Scene + Phasor Diagram

def plot_scene(f_MHz, d_m, h_tx_m, h_rx_m, Pt_dBm, refl_coeff):
    f_Hz = f_MHz * 1e6
    res = two_path_power(f_Hz, d_m, h_tx_m, h_rx_m, Pt_dBm, refl_coeff=refl_coeff)
    Prx_two, Prx_los, phase_diff, d1, d2, E1, E2, phi1, phi2 = res

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5),
                              gridspec_kw={'width_ratios': [2, 1]})
    fig.patch.set_facecolor('#f5f5f5')

    # LEFT: geometry
    ax = axes[0]
    ax.set_facecolor('#e8f4fb')
    ax.set_xlim(-0.5, d_m + 1)
    y_max = max(h_tx_m, h_rx_m) * 2.2
    ax.set_ylim(-0.3, y_max)
    ax.set_xlabel('Horizontal distance [m]', fontsize=10)
    ax.set_ylabel('Height [m]', fontsize=10)
    ax.set_title(f'Two-Path Model Geometry   |   f = {f_MHz:.0f} MHz   |   d = {d_m:.0f} m',
                 fontsize=11, fontweight='bold')

    # Ground
    ax.axhline(0, color='#8B6914', lw=3, zorder=2)
    ax.fill_between([0, d_m], -0.3, 0, color='#c8a96e', alpha=0.5)
    ax.text(d_m/2, -0.22, 'Ground (reflective surface)', ha='center', fontsize=8,
            color='#6B4F12')

    # Tx antenna
    ax.annotate('', xy=(0, h_tx_m), xytext=(0, 0),
                arrowprops=dict(arrowstyle='-', color='#555', lw=2))
    ax.plot([-0.15, 0.15], [h_tx_m, h_tx_m], 'k-', lw=3, zorder=5)
    ax.plot([-0.12, 0.12], [h_tx_m-0.04*y_max, h_tx_m-0.04*y_max], 'k-', lw=2, zorder=5)
    ax.text(-0.3, h_tx_m*0.5, f'h_Tx\n{h_tx_m:.0f}m', ha='center', fontsize=8, color='#333')
    ax.text(0, h_tx_m + 0.05*y_max, 'Tx', ha='center', fontsize=9, fontweight='bold', color='#2c3e50')

    # Rx antenna
    ax.annotate('', xy=(d_m, h_rx_m), xytext=(d_m, 0),
                arrowprops=dict(arrowstyle='-', color='#555', lw=2))
    ax.plot([d_m-0.15, d_m+0.15], [h_rx_m, h_rx_m], 'k-', lw=3, zorder=5)
    ax.plot([d_m-0.12, d_m+0.12], [h_rx_m-0.04*y_max, h_rx_m-0.04*y_max], 'k-', lw=2, zorder=5)
    ax.text(d_m+0.35, h_rx_m*0.5, f'h_Rx\n{h_rx_m:.0f}m', ha='center', fontsize=8, color='#333')
    ax.text(d_m, h_rx_m + 0.05*y_max, 'Rx', ha='center', fontsize=9, fontweight='bold', color='#2c3e50')

    # Direct path (green)
    ax.annotate('', xy=(d_m, h_rx_m), xytext=(0, h_tx_m),
                arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2.5,
                                mutation_scale=15))
    mid_x = d_m/2; mid_y = (h_tx_m + h_rx_m)/2
    ax.text(mid_x, mid_y + 0.06*y_max, f'Direct  d₁={d1:.1f}m',
            ha='center', fontsize=8.5, color='#27ae60',
            bbox=dict(fc='white', ec='#27ae60', alpha=0.85, pad=2, boxstyle='round'))

    # Ground reflection point
    refl_x = d_m * h_tx_m / (h_tx_m + h_rx_m)
    ax.plot(refl_x, 0, 'o', ms=8, color='#e74c3c', zorder=6)

    # Reflected path (red, two segments)
    ax.annotate('', xy=(refl_x, 0), xytext=(0, h_tx_m),
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2.5,
                                mutation_scale=15))
    ax.annotate('', xy=(d_m, h_rx_m), xytext=(refl_x, 0),
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2.5,
                                mutation_scale=15))
    ax.text(refl_x/2 + 0.03*d_m, h_tx_m*0.4,
            f'Reflected  d₂={d2:.1f}m',
            ha='center', fontsize=8.5, color='#e74c3c',
            bbox=dict(fc='white', ec='#e74c3c', alpha=0.85, pad=2, boxstyle='round'))

    # Breakpoint marker
    d_bp = breakpoint_distance(f_Hz, h_tx_m, h_rx_m)
    if d_bp < d_m:
        ax.axvline(d_bp, color='#f39c12', lw=2, linestyle='--', zorder=3)
        ax.text(d_bp + 0.01*d_m, 0.05*y_max, f'Breakpunkt\n{d_bp:.0f}m',
                fontsize=8, color='#f39c12',
                bbox=dict(fc='white', ec='#f39c12', alpha=0.85, pad=2, boxstyle='round'))

    # Info box
    phase_deg = np.degrees(phase_diff)
    if abs(np.cos(phase_diff)) > 0.5:
        interf = 'Constructive ✅'; ic = '#27ae60'
    else:
        interf = 'Destructive ⚠️'; ic = '#e74c3c'

    ax.text(d_m*0.98, y_max*0.97,
            f'Phase diff: {phase_deg:.1f}°\n{interf}\nP_Rx(2-path)={Prx_two:.1f} dBm\nP_Rx(LOS)  ={Prx_los:.1f} dBm',
            ha='right', va='top', fontsize=9,
            bbox=dict(fc='white', ec=ic, alpha=0.92, pad=5, boxstyle='round,pad=0.4'),
            color='#2c3e50')
    ax.grid(True, linestyle=':', alpha=0.3)

    # RIGHT: Phasor diagram
    ax2 = axes[1]
    ax2.set_facecolor('#f0f4fa')
    ax2.set_aspect('equal')
    ax2.set_xlim(-1.6, 1.6); ax2.set_ylim(-1.6, 1.6)
    ax2.set_title('Phasor Diagram at Rx', fontsize=11, fontweight='bold')
    ax2.axhline(0, color='#bbb', lw=1); ax2.axvline(0, color='#bbb', lw=1)
    ax2.grid(True, linestyle=':', alpha=0.3)

    scale = 1.0 / (E1 + E2 + 1e-20)
    px1 = E1 * np.cos(phi1) * scale; py1 = E1 * np.sin(phi1) * scale
    px2 = E2 * np.cos(phi2) * scale; py2 = E2 * np.sin(phi2) * scale
    px_t = (E1*np.exp(1j*phi1) + E2*np.exp(1j*phi2)).real * scale
    py_t = (E1*np.exp(1j*phi1) + E2*np.exp(1j*phi2)).imag * scale

    ax2.annotate('', xy=(px1, py1), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='#27ae60', lw=3, mutation_scale=18))
    ax2.annotate('', xy=(px1+px2, py1+py2), xytext=(px1, py1),
                 arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=3, mutation_scale=18))
    ax2.annotate('', xy=(px_t, py_t), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='#8e44ad', lw=3.5, mutation_scale=18))

    ax2.text(px1*0.55, py1*0.55 + 0.06, 'E_direct', fontsize=8.5, color='#27ae60',
             ha='center')
    ax2.text(px1+px2*0.55, py1+py2*0.55, 'E_reflected', fontsize=8.5, color='#e74c3c',
             ha='center')
    ax2.text(px_t*0.55, py_t*0.55 - 0.1, 'E_total', fontsize=9, color='#8e44ad',
             ha='center', fontweight='bold')

    legend_items = [
        mpatches.Patch(color='#27ae60', label='Direct'),
        mpatches.Patch(color='#e74c3c', label='Reflected'),
        mpatches.Patch(color='#8e44ad', label='Total'),
    ]
    ax2.legend(handles=legend_items, fontsize=8, loc='lower right')

    plt.tight_layout()
    plt.show()

print("✅ Scene plotter defined.")


In [ ]:
# VISUALISATION 2 — Received Power vs. Distance

def plot_power_vs_distance(f_MHz, h_tx_m, h_rx_m, Pt_dBm, d_highlight):
    f_Hz = f_MHz * 1e6
    distances = np.logspace(1, 4, 3000)   # 10 m ... 10 km

    P_two = []; P_los = []
    for d in distances:
        pt, pl, *_ = two_path_power(f_Hz, d, h_tx_m, h_rx_m, Pt_dBm)
        P_two.append(pt); P_los.append(pl)
    P_two = np.array(P_two); P_los = np.array(P_los)

    d_bp = breakpoint_distance(f_Hz, h_tx_m, h_rx_m)

    fig, ax = plt.subplots(figsize=(11, 5.5))
    fig.patch.set_facecolor('#f5f5f5')
    ax.set_facecolor('#eaf4fb')

    ax.semilogx(distances, P_two, color='#e74c3c', lw=2.5, label='Two-path model', zorder=4)
    ax.semilogx(distances, P_los, color='#2980b9', lw=2, linestyle='--',
                label='Direct path only (Friis)', zorder=3)

    # Reference slopes
    ref_val = P_los[100]
    d_ref = distances[100]
    slope20 = ref_val - 20*(np.log10(distances) - np.log10(d_ref))
    anchor_idx = np.argmin(np.abs(distances - d_bp*1.5)) if d_bp < distances[-1] else -200
    anchor_d = d_bp*1.5 if d_bp < distances[-1] else distances[-200]
    slope40 = P_two[anchor_idx] - 40*(np.log10(distances) - np.log10(anchor_d))
    ax.semilogx(distances, slope20, 'k--', lw=1.2, alpha=0.4, label='20 dB/decade ref.')
    ax.semilogx(distances, slope40, 'k:',  lw=1.2, alpha=0.4, label='40 dB/decade ref.')

    # Breakpoint
    if 10 < d_bp < 1e4:
        ax.axvline(d_bp, color='#f39c12', lw=2, linestyle='-.', zorder=5)
        y0 = ax.get_ylim()[0]
        ax.text(d_bp*1.05, y0*0.85 if y0 < 0 else -120,
                f'Breakpunkt\n{d_bp:.0f} m', fontsize=8.5, color='#f39c12',
                bbox=dict(fc='white', ec='#f39c12', alpha=0.88, pad=2, boxstyle='round'))

    # Highlight current distance
    pt_hl, pl_hl, *_ = two_path_power(f_Hz, d_highlight, h_tx_m, h_rx_m, Pt_dBm)
    ax.axvline(d_highlight, color='#8e44ad', lw=2, linestyle=':', zorder=6)
    ax.plot(d_highlight, pt_hl, 'o', ms=10, color='#8e44ad', zorder=7,
            label=f'Current d={d_highlight:.0f}m → {pt_hl:.1f} dBm')

    ax.set_xlabel('Distance [m]', fontsize=11)
    ax.set_ylabel('Received power [dBm]', fontsize=11)
    ax.set_title(f'Two-Path Model: P_Rx vs. Distance\n'
                 f'f={f_MHz:.0f} MHz | h_Tx={h_tx_m:.0f}m | h_Rx={h_rx_m:.0f}m | P_Tx={Pt_dBm:.0f} dBm',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, which='both', linestyle=':', alpha=0.4)
    ax.set_xlim(10, 1e4)

    plt.tight_layout()
    plt.show()

print("✅ Power vs. distance plotter defined.")


In [ ]:
# VISUALISATION 3 — Wave Superposition Animation

def plot_wave_superposition(f_MHz, phase_diff_deg):
    phase_diff = np.radians(phase_diff_deg)
    x = np.linspace(0, 4*np.pi, 600)

    E1 = np.cos(x)
    E2 = 0.8 * np.cos(x + phase_diff)
    E_total = E1 + E2

    fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
    fig.patch.set_facecolor('#f5f5f5')

    labels = ['E_direct (normalized)', 'E_reflected (Γ×normalized)', 'E_total (superposition)']
    waves  = [E1, E2, E_total]
    colors = ['#27ae60', '#e74c3c', '#8e44ad']

    for ax, w, lbl, col in zip(axes, waves, labels, colors):
        ax.set_facecolor('#eaf4fb')
        ax.plot(x / np.pi, w, color=col, lw=2.5, label=lbl)
        ax.fill_between(x / np.pi, w, alpha=0.15, color=col)
        ax.axhline(0, color='#999', lw=1)
        ax.set_ylim(-2.2, 2.2)
        ax.set_ylabel('Amplitude', fontsize=9)
        ax.legend(fontsize=9, loc='upper right')
        ax.grid(True, linestyle=':', alpha=0.3)

    axes[-1].set_xlabel('Phase [× π]', fontsize=10)

    pd = phase_diff_deg % 360
    if pd < 45 or pd > 315:
        kind = '→ Constructive interference 📈'; c = '#27ae60'
    elif 135 < pd < 225:
        kind = '→ Destructive interference 📉'; c = '#e74c3c'
    else:
        kind = '→ Partial interference'; c = '#f39c12'

    fig.suptitle(f'Wave Superposition at Receiver   |   Phase diff = {phase_diff_deg:.0f}°   |   {kind}',
                 fontsize=11, fontweight='bold', color=c)
    plt.tight_layout()
    plt.show()

print("✅ Wave superposition plotter defined.")


In [ ]:
# INTERACTIVE EXPLORER — all three plots in one widget

style  = {'description_width': '160px'}
layout = widgets.Layout(width='450px')

sl_freq   = widgets.FloatSlider(value=1800, min=700, max=6000, step=100,
                description='Frequency [MHz]:', style=style, layout=layout, readout_format='.0f')
sl_dist   = widgets.FloatLogSlider(value=500, base=10, min=1, max=4, step=0.01,
                description='Distance d [m]:', style=style, layout=layout, readout_format='.0f')
sl_htx    = widgets.FloatSlider(value=30,  min=5,   max=200, step=1,
                description='h_Tx [m]:', style=style, layout=layout, readout_format='.0f')
sl_hrx    = widgets.FloatSlider(value=1.5, min=0.5, max=20,  step=0.5,
                description='h_Rx [m]:', style=style, layout=layout, readout_format='.1f')
sl_pt     = widgets.FloatSlider(value=30,  min=0,   max=50,  step=1,
                description='P_Tx [dBm]:', style=style, layout=layout, readout_format='.0f')
sl_refl   = widgets.FloatSlider(value=-1.0, min=-1.0, max=0.0, step=0.05,
                description='Γ (refl. coeff):', style=style, layout=layout, readout_format='.2f')

out = widgets.Output()

def update(_=None):
    with out:
        clear_output(wait=True)
        f    = sl_freq.value * 1e6
        d    = sl_dist.value
        htx  = sl_htx.value
        hrx  = sl_hrx.value
        pt   = sl_pt.value
        refl = sl_refl.value

        pt2, plos, phase_diff, d1, d2, E1, E2, phi1, phi2 = two_path_power(f, d, htx, hrx, pt, refl_coeff=refl)

        d_bp = breakpoint_distance(f, htx, hrx)
        phase_deg = np.degrees(phase_diff) % 360

        line = '━' * 54
        print(line)
        print(f"  λ = {c0/f*100:.2f} cm   |   d₁ (direct) = {d1:.2f} m   |   d₂ (reflect) = {d2:.2f} m")
        print(f"  Phase difference = {phase_deg:.1f}°  |  P_Rx (2-path) = {pt2:.1f} dBm  |  P_Rx (LOS) = {plos:.1f} dBm")
        zone = '4th-order roll-off zone (40 dB/dec)' if d > d_bp else '2nd-order zone (20 dB/dec)'
        rel  = '>' if d > d_bp else '<'
        print(f"  Breakpunkt = {d_bp:.0f} m  |  d {rel} d_BP  →  {zone}")
        print(line)

        plot_scene(sl_freq.value, d, htx, hrx, pt, refl)
        plot_power_vs_distance(sl_freq.value, htx, hrx, pt, d)
        plot_wave_superposition(sl_freq.value, phase_deg)

for sl in [sl_freq, sl_dist, sl_htx, sl_hrx, sl_pt, sl_refl]:
    sl.observe(update, names='value')

hint = (
    "<h3>⚙️ Two-Path Model — Interactive Parameters</h3>"
    "<p style='color:#555;font-size:13px'>"
    "Adjust the sliders to explore how antenna heights, distance, frequency, and ground "
    "reflectivity affect the received power and the interference pattern.<br>"
    "<b>Breakpunkt</b> = 4·h_Tx·h_Rx / λ — beyond this the power drops at <b>40 dB/decade</b>.</p>"
)

ui = widgets.VBox([
    widgets.HTML(hint),
    widgets.HBox([
        widgets.VBox([sl_freq, sl_dist, sl_htx]),
        widgets.VBox([sl_hrx, sl_pt, sl_refl])
    ]),
    out
])
display(ui)
update()


## 📚 Theory Reference

### Two-Path Model (Ground Reflection)

The total received field is the superposition of the direct and reflected waves:

$$E_{total} = E_1 \cdot e^{j\phi_1} + \Gamma \cdot E_2 \cdot e^{j\phi_2}$$

where:
- \(E_1 = \frac{\lambda}{4\pi d_1}\), \(E_2 = \frac{\lambda}{4\pi d_2}\) — free-space field amplitudes
- \(\Gamma = -1\) for a perfect ground reflection (phase flip of 180°)
- \(d_1 = \sqrt{d^2 + (h_{Tx} - h_{Rx})^2}\) — direct path
- \(d_2 = \sqrt{d^2 + (h_{Tx} + h_{Rx})^2}\) — reflected path (image method)

### Interference Condition

$$\Delta\phi = \frac{2\pi}{\lambda}(d_2 - d_1) + \pi$$

| Condition | Result |
|-----------|--------|
| \(\Delta\phi = 2n\pi\) | **Constructive** → signal boost |
| \(\Delta\phi = (2n+1)\pi\) | **Destructive** → signal cancellation |

### Breakpoint Distance

$$d_{BP} = \frac{4 \cdot h_{Tx} \cdot h_{Rx}}{\lambda}$$

- For \(d < d_{BP}\): oscillations, average slope ≈ 20 dB/decade (like free space)
- For \(d > d_{BP}\): constructive/destructive merge → **40 dB/decade** roll-off

### Simplified Far-Field Formula (d >> d_BP)

$$P_{Rx} \approx P_{Tx} \cdot G_{Tx} \cdot G_{Rx} \cdot \left(\frac{h_{Tx} \cdot h_{Rx}}{d^2}\right)^2$$

> **Key insight:** No frequency dependence in the far field! Power falls as \(d^{-4}\).
